In [8]:
import pandas as pd
df = pd.read_csv('physic.csv')

In [9]:
import pandas as pd
from pathlib import Path

# Đổi path này theo file của bạn
INPUT_CSV = "physic.CSV"

OUTPUT_DIR = Path("split_output")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_CSV)

# Lấy 2 chữ đầu của id: LD005 -> LD, TD402 -> TD
df["type"] = df["id"].astype(str).str[:2]

train_parts = []
test_parts = []

RANDOM_SEED = 42

for type_name, group in df.groupby("type", sort=False):
    # Shuffle từng nhóm
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    
    n = len(group)

    if n == 1:
        # Không thể chia 80/20 với 1 dòng, cho vào train
        n_train = 1
    else:
        # Đảm bảo nhóm có >=2 dòng thì test có ít nhất 1 dòng
        n_test = max(1, round(n * 0.2))
        n_train = n - n_test

    train_parts.append(group.iloc[:n_train])
    test_parts.append(group.iloc[n_train:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

# Xóa cột phụ "type" trước khi save nếu không muốn giữ
train_df.drop(columns=["type"]).to_csv(OUTPUT_DIR / "80.csv", index=False, encoding="utf-8-sig")
test_df.drop(columns=["type"]).to_csv(OUTPUT_DIR / "20.csv", index=False, encoding="utf-8-sig")

print("Original:", len(df))
print("80.csv:", len(train_df))
print("20.csv:", len(test_df))

print("\nOriginal type counts:")
print(df["type"].value_counts().sort_index())

print("\n80.csv type counts:")
print(train_df["type"].value_counts().sort_index())

print("\n20.csv type counts:")
print(test_df["type"].value_counts().sort_index())

Original: 1351
80.csv: 1081
20.csv: 270

Original type counts:
type
CH    310
DD    130
DT     68
LD    396
NL    190
TD    177
TH     80
Name: count, dtype: int64

80.csv type counts:
type
CH    248
DD    104
DT     54
LD    317
NL    152
TD    142
TH     64
Name: count, dtype: int64

20.csv type counts:
type
CH    62
DD    26
DT    14
LD    79
NL    38
TD    35
TH    16
Name: count, dtype: int64


In [10]:
import re
import numpy as np
import pandas as pd

# df columns: id, question, cot, answer, unit
df = pd.read_csv('physic.csv')
NUM_PATTERN = r'[-+]?\d+(?:\.\d+)?(?:\s*(?:×|x|\*)\s*10\^[-+]?\d+|[eE][-+]?\d+)?'

def parse_number(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip()

    # lấy số đầu tiên trong answer hoặc số cuối trong text tùy hàm gọi
    m = re.search(NUM_PATTERN, s)
    if not m:
        return np.nan

    num = m.group(0)
    num = re.sub(r'\s+', '', num)
    num = num.replace('×10^', 'e')
    num = num.replace('x10^', 'e')
    num = num.replace('*10^', 'e')

    try:
        return float(num)
    except ValueError:
        return np.nan


def extract_last_number_from_cot(cot):
    if pd.isna(cot):
        return np.nan

    s = str(cot)
    matches = re.findall(NUM_PATTERN, s)

    if not matches:
        return np.nan

    num = matches[-1]
    num = re.sub(r'\s+', '', num)
    num = num.replace('×10^', 'e')
    num = num.replace('x10^', 'e')
    num = num.replace('*10^', 'e')

    try:
        return float(num)
    except ValueError:
        return np.nan


def extract_last_unit_from_cot(cot):
    if pd.isna(cot):
        return ""

    s = str(cot)

    # ưu tiên unit ở gần cuối câu
    unit_pattern = r'\b(N|C|V|A|Ω|ohm|J|W|m|cm|mm|kg|s|Hz|F)\b'
    matches = re.findall(unit_pattern, s)

    if not matches:
        return ""

    return matches[-1]


def normalize_unit(unit):
    if pd.isna(unit):
        return ""

    u = str(unit).strip()
    u = u.replace("ohm", "Ω")

    return u


df = df.copy()

df["cot_final_value"] = df["cot"].apply(extract_last_number_from_cot)
df["answer_value"] = df["answer"].apply(parse_number)

df["cot_final_unit"] = df["cot"].apply(extract_last_unit_from_cot).apply(normalize_unit)
df["answer_unit"] = df["unit"].apply(normalize_unit)

df["value_match"] = np.isclose(
    df["cot_final_value"],
    df["answer_value"],
    rtol=1e-3,
    atol=1e-9,
    equal_nan=False
)

df["unit_match"] = df["cot_final_unit"] == df["answer_unit"]

df["match"] = df["value_match"] & df["unit_match"]

mismatch_df = df[~df["match"]].copy()

mismatch_df.to_csv("cot_answer_unit_mismatch.csv", index=False, encoding="utf-8-sig")

print("Total rows:", len(df))
print("Matched:", df["match"].sum())
print("Mismatched:", len(mismatch_df))

mismatch_df[[
    "id",
    "cot_final_value",
    "answer_value",
    "cot_final_unit",
    "answer_unit",
    "value_match",
    "unit_match"
]].head(20)

Total rows: 1351
Matched: 600
Mismatched: 751


,id,cot_final_value,answer_value,cot_final_unit,answer_unit,value_match,unit_match
1,TD402,1.000000e+02,1.000000e+02,F,μ,True,False
2,LD001,5.000000e-02,5.000000e-02,C,N,True,False
6,LD005,7.000000e+00,9.000000e+00,C,N,False,False
15,LD014,3.460000e+00,3.460000e+00,C,μC,True,False
18,LD017,3.200000e+01,3.600000e+00,N,N,False,True
21,LD020,1.200000e+02,1.200000e+02,N,degree,True,False
24,LD023,1.500000e+01,6.000000e+00,N,N,False,True
26,LD025,2.300000e+01,6.250000e+00,N,N,False,True
42,LD041,0.000000e+00,2.000000e+00,F,N,False,False
48,LD047,2.000000e+00,NaN,cm,-,False,False
